# DAY 09 -- Infinite Memory (Generators)

### MC9.1 : The Basic Yield

In [2]:
# Write a function `gen()` that yields `1`, then `2`, then `3`. Loop through it.
def gen():
    yield from (1, 2, 3)

for x in gen():
    print(x)


1
2
3


> **Deep Dive:** Unlike `return`, `yield` pauses the function and saves the **Stack Frame** (local variables) in RAM. The function is “frozen” until you call it again.

---

### MC9.2 : The Memory Profile


In [13]:
# Compare `sys.getsizeof()` of a list comprehension vs a generator expression for 1 million numbers.
import sys

print(
    "Size of list:  ", sys.getsizeof(list(range(1_000_000))),
    "\nSize of generator: ", sys.getsizeof((x for x in range(1_000_000))),
)


Size of list:   8000056 
Size of generator:  104


> **Deep Dive:** 
> `[x for x in range(1M)]` consumes ~8MB RAM (stores all numbers).
> `(x for x in range(1M))` consumes ~100 bytes (stores only the logic).


---


### MC9.3 : The Infinite Sequence


In [15]:
# Write a `while True` generator that produces Fibonacci numbers forever.
def fib():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

g = fib()
for _ in range(5):
    print(next(g))

print(next(g), next(g), next(g), "...", sep=", ")  # Show that it continues

0
1
1
2
3
5, 8, 13, ...


> This is impossible with a standard list (RAM would fill up). Generators allow infinite data streams by processing one item at a time (lazy evaluation).


---


### MC9.4 : The One-Time Trap


In [16]:
# Create a generator `g`. Loop through it once. Try to loop through it again.
g = (x for x in range(3))
print(list(g))
print(list(g))


print("-----", list(x for x in range(3))) # recreating the generator works

[0, 1, 2]
[]
----- [0, 1, 2]


> **Deep Dive:** 
> Generators are exhaustible. Once iterated, the "cursor" is at the end. You cannot rewind a generator; you must re-instantiate it.


---


### MC9.5 : The Next Protocol


In [19]:
# Manually call `next(gen)` until it crashes. ## [Safely handle the exception.]
g = iter(range(3))
while True:
    try:
        print(next(g))
    except StopIteration as e:
        print(f"Generator exhausted: {e!r}")
        break


0
1
2
Generator exhausted: StopIteration()


In [ ]:
g = iter(range(3))
while True: print(next(g)) # This will eventually raise StopIteration

0
1
2


StopIteration: 

> **Deep Dive:** 
> When a generator runs out of items, it raises `StopIteration` exception. `for` loops catch this exception silently to stop looping.


---


### MC9.6 : The Pipeline (Chaining)


In [7]:
# Create two generators: one squares numbers, the other filters evens. Chain them: `filter(square(nums))`.
nums = range(6)

def square(it):
    for n in it:
        yield n * n

def even(it):
    for n in it:
        if n % 2 == 0:
            yield n

print(*even(square(nums)))


0 4 16


> **Deep Dive:** 
> This creates a data pipeline: source -> square -> filter, one item at a time. No intermediate lists are created in RAM.


---


### MC9.7 : The Large File Reader


In [8]:
# Write a generator to read a "fake" 100GB file line-by-line.
def fake_file(n):
    for i in range(n):
        yield f"line {i}"

for line in fake_file(3):
    print(line)


line 0
line 1
line 2


> **Deep Dive:** 
> Using `yield line` lets you process datasets larger than RAM. This is the standard for big data processing in Python.


---


### MC9.8 : Yield From


In [9]:
# Write a generator that yields values from two sub-generators using `yield from`.
def g1():
    yield from (1, 2)

def g2():
    yield from (3, 4)

def g():
    yield from g1()
    yield from g2()

print(list(g()))


[1, 2, 3, 4]


> **Deep Dive:** 
> `yield from` delegates to a sub-generator and flattens nested iteration without manual loops.


---


### MC9.9 : The Send Method


In [10]:
# Use `gen.send(value)` to inject data into a running generator.
def echo():
    val = yield
    while True:
        val = yield val

g = echo()
next(g)
print(g.send(10))
print(g.send(20))


10
20


> **Deep Dive:** 
> Generators can be two-way streets. `val = yield` pauses and waits to receive data. This is the basis for coroutines (AsyncIO).


---


### MC9.10 : State Retention


In [22]:
# Write a generator that calculates a running average.
def running_avg():
    total = count = 0
    while True:
        x = yield total / count if count else 0
        total += x
        count += 1

g = running_avg()
next(g) # Prime the generator -- can only be done once
print("Average after sending 10:", g.send(10))
print("Average after sending 20:", g.send(20))
print("Average after sending 30:", g.send(30))

## print("Current average without new input:", next(g)) # raises TypeError


Average after sending 10: 10.0
Average after sending 20: 15.0
Average after sending 30: 20.0


> **Deep Dive:** 
> The generator remembers `total` and `count` between yields, avoiding globals or class state.


---
